# ClauseLens · Phase 4 · Transformer fine-tuning (Colab)

**Pehle:** `Runtime → Change runtime type → T4 GPU`, phir cells upar se neeche ek-ek karke chala.

Is notebook mein: code upload → data build (CUAD + LEDGAR + synthetic) → TF-IDF baselines → legal-BERT / InLegalBERT / bert-base fine-tune → comparison table → demo → Drive pe save.

Selection rule: best epoch **sirf `real_val`** pe chuna jata hai. `synth_test` aur `gold` kabhi selection mein use nahi hote.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NOT FOUND -> Runtime > Change runtime type > T4 GPU")

## 1. Code upload
`clauselens-complete.zip` upload kar (file picker khulega). Drive se lena ho to: `drive.mount(...)` karke zip ko `/content` mein unzip kar.

In [ ]:
import os, zipfile
from google.colab import files

if not os.path.exists("/content/clauselens"):
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall("/content")
%cd /content/clauselens

In [ ]:
!pip install -q -e ".[nlp]"

## Tests (torch wale bhi)
Training se pehle poora test-suite chala. `test_finetune.py` tiny model pe fine-tuning code check karta hai; ye pass na ho to aage mat badh.


In [ ]:
!pip install -q pytest httpx
!python -m pytest -rs


## 2. Data build
CUAD (GitHub se, checksum verified), LEDGAR (HuggingFace), phir clean dataset + synthetic rent/bond data.

In [ ]:
!python scripts/download_cuad.py
!python scripts/download_ledgar.py
!python scripts/build_dataset_v1.py
!python scripts/generate_synthetic.py

## 3. TF-IDF baselines (comparison ke liye)

In [ ]:
!python scripts/train_baselines.py 2>&1 | grep -v -i warn

## 4. Smoke run (1 minute) — pehle pipeline check, phir poora training
Ye `/tmp` mein likhta hai, asli reports ko nahi chhuta.

In [ ]:
!python scripts/train_transformer.py --model-name nlpaueb/legal-bert-base-uncased --epochs 1 --max-train-examples 300 --models-dir /tmp/smoke_models --reports-dir /tmp/smoke_reports

## 5. Poora fine-tuning
Teen models: **legal-BERT** (legal domain), **InLegalBERT** (Indian court judgments), **bert-base-uncased** (general control).
`nlpaueb/bert-base-uncased-contracts` bhi try kar sakta hai (contracts pe pretrain hai).

In [ ]:
MODELS = ["nlpaueb/legal-bert-base-uncased", "law-ai/InLegalBERT", "bert-base-uncased"]
for m in MODELS:
    get_ipython().system(f"python scripts/train_transformer.py --model-name {m} --epochs 4")

## 6. Comparison table

In [ ]:
!python scripts/compare_reports.py

## 7. Demo: poora document -> segments -> clause types
`MODEL_DIR` badal ke dusra model try kar. Apna agreement text `MY_TEXT` mein paste kar sakta hai.

In [ ]:
import pandas as pd
from clauselens.inference import ClauseClassifier, classify_document
from clauselens.synth.generator import generate_documents

MODEL_DIR = "models/nlpaueb__legal-bert-base-uncased"
clf = ClauseClassifier(MODEL_DIR)

MY_TEXT = None   # apna agreement text yahan paste kar (string), warna synthetic doc use hoga
text = MY_TEXT or generate_documents(1, seed=123)[0]["text"]
rows = classify_document(text, clf)
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows)[["index", "kind", "heading", "labels", "top", "text"]]

## 8. Drive pe save
Colab session khatam hote hi `/content` udd jaata hai, isliye models aur reports Drive mein copy kar.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/clauselens_outputs
!cp -r models reports /content/drive/MyDrive/clauselens_outputs/

**Gold set (jab ready ho):** `gold_v1.jsonl` ko `data/gold/` mein upload kar aur section 3 + 5 dobara chala, `gold` column apne aap aa jayega.